# Fine-Tuning Comparison

Train SFT+LoRA, SFT+QLoRA, DPO, Reward Model, and GRPO on the StackOverflow Q&A dataset.

### Imports & Seed

In [ ]:
# GLOBAL QUIET MODE
import os, sys, logging, warnings, importlib

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]      = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"]          = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"]            = "error"
os.environ["ACCELERATE_LOG_LEVEL"]              = "error"
os.environ["BITSANDBYTES_NOWELCOME"]            = "1"
os.environ["TOKENIZERS_PARALLELISM"]            = "false"
os.environ["TRL_DISABLE_RICH"]                  = "1"   # hides rich tables/progress
os.environ["TORCHAO_DISABLE_CPP_EXTENSIONS"]    = "1"
os.environ["TORCHAO_SKIP_CPP_LOAD"]             = "1"
os.environ["USE_TORCHAO"]                       = "0"
os.environ['HF_TOKEN'] = 'hf_...'   # your huggingface token

In [ ]:
import sys
sys.path.append('..')

import torch
import pandas as pd
from datasets import Dataset
from src.utils.config_loader import load_config
from pathlib import Path
from src.utils.seed import set_seed
from src.data.preprocess import (
    load_stackoverflow, split_qa,
    make_sft_dataframe, make_preference_pairs,
    make_reward_pairs, make_grpo_prompts,
)
from src.utils.version import banner

set_seed(42)

### Load and Split Data

In [ ]:
base_cfg = load_config("configs/base.yaml")
set_seed(base_cfg["seed"])

# KAGGLE_DATA_DIR override so the same YAML works on Kaggle and locally.
DATA_DIR = "/a-survey-of-LLMs-fine-tuning-approaches/data/stackoverflow/stacksample"

df = load_stackoverflow(DATA_DIR)
train_df, val_df, test_df = split_qa(df, seed=base_cfg["seed"])
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print("Environment:", banner())

### SFT + LoRA

In [ ]:
from src.training.sft import train_sft

cfg = load_config("configs/sft_lora.yaml")

sft_train = Dataset.from_pandas(make_sft_dataframe(train_df), preserve_index=False)
sft_val   = Dataset.from_pandas(make_sft_dataframe(val_df),   preserve_index=False)

sft_lora_trainer = train_sft(cfg, sft_train, sft_val)
print("Saved SFT+LoRA to:", cfg["output_dir"])

### SFT + QLoRA

In [ ]:
from src.training.sft import train_sft

cfg = load_config("configs/sft_qlora.yaml")

sft_train = Dataset.from_pandas(make_sft_dataframe(train_df), preserve_index=False)
sft_val   = Dataset.from_pandas(make_sft_dataframe(val_df),   preserve_index=False)

sft_qlora_trainer = train_sft(cfg, sft_train, sft_val)
print("Saved SFT+QLoRA to:", cfg["output_dir"])

### DPO

In [ ]:
from src.training.dpo import train_dpo

cfg = load_config("configs/dpo.yaml")

dpo_df = make_preference_pairs(train_df)
dpo_train, dpo_val = make_train_val_datasets(
    dpo_df, frac=0.8, seed=cfg.get("seed", 42)
)

dpo_trainer = train_dpo(cfg, dpo_train, dpo_val)
print("Saved DPO to:", cfg["output_dir"])

### Reward Modeling

In [ ]:
from src.training.reward import train_reward 

cfg = load_config("configs/reward.yaml")

rew_df = make_reward_pairs(train_df)
rew_train, rew_val = make_train_val_datasets(
    rew_df, frac=0.8, seed=cfg.get("seed", 42)
)

reward_trainer = train_reward(cfg, rew_train, rew_val)
print("Saved Reward to:", cfg["output_dir"])

### GRPO

In [ ]:
from src.training.grpo import train_grpo

cfg = load_config("configs/grpo.yaml")

grpo_df = make_grpo_prompts(train_df)
grpo_train, grpo_val = make_train_val_datasets(
    grpo_df, frac=0.8, seed=cfg.get("seed", 42)
)

grpo_trainer = train_grpo(cfg, grpo_train, grpo_val)
print("Saved GRPO to:", cfg["output_dir"])

### Comparison Table

In [ ]:
rows = []
for name in ["sft_lora", "sft_qlora", "dpo", "reward", "grpo"]:
    cfg = load_config(f"configs/{name}.yaml")
    path = cfg["output_dir"]
    exists = os.path.isdir(path)
    files = sorted(os.listdir(path)) if exists else []
    rows.append({
        "method": name,
        "output_dir": path,
        "exists": exists,
        "n_files": len(files),
    })

summary = pd.DataFrame(rows)
summary